# Import Library

In [ ]:
from kaggle_secrets import UserSecretsClient
from openai import OpenAI
import pandas as pd
import json
import time
from tqdm.auto import tqdm
from concurrent.futures import ThreadPoolExecutor

user_secrets = UserSecretsClient()

# Load Dataset

In [ ]:
df = pd.read_csv('/kaggle/input/result-lawbot-mistral/result_generate.csv')
print(f"Shape: {df.shape}")
df.head()

In [ ]:
# Sampling 2 row
sampling_df = df.sample(n=3, random_state=123)
sampling_df.head()

# Prompt LLM Jugde

In [ ]:
prompt_template = prompt_template = """Anda adalah evaluator AI yang tegas. Tugas Anda adalah menilai kualitas `Generate Answer` dari sebuah chatbot hukum. Evaluasi dilakukan dengan membandingkannya terhadap `Questions` dari pengguna dan `Reference Answer` yang ideal.

Terdapat dua tujuan evaluasi utama:
1.  **Relevansi**: Apakah `Generate Answer` menjawab `Questions` dengan tepat?
2.  **Kesesuaian**: Seberapa akurat `Generate Answer` mencerminkan isi dan kutipan hukum dari `Reference Answer`?

**INPUT:**
- `Questions`: "{question}"
- `Reference Answer`: "{reference_answer}"
- `Generate Answer`: "{generate_answer}"

**TUGAS:**
Isi objek JSON berikut dengan evaluasi Anda. Berikan skor dari 1 (sangat buruk) hingga 5 (sangat baik). Alasan harus ringkas dan dalam Bahasa Indonesia.

**OUTPUT (HANYA JSON):**
```json
{{
  "relevansi": {{
    "skor": <1-5>,
    "alasan": "<Apakah jawaban menjawab 'Questions'? Mengapa?>"
  }},
  "akurasi_terhadap_sumber": {{
    "skor": <1-5>,
    "alasan": "<Apakah secara faktual sejalan dengan 'Reference Answer'?>"
  }},
  "kelengkapan_terhadap_sumber": {{
    "skor": <1-5>,
    "alasan": "<Apakah mencakup poin-poin kunci dari 'Reference Answer'?>"
  }},
  "akurasi_kutipan_pasal": {{
    "penilaian": "<Sangat Akurat | Akurat Sebagian | Tidak Akurat | Tidak Menyebutkan>",
    "alasan": "<Bandingkan kutipan: Hasil [pasal] vs. Referensi [pasal].>"
  }},
  "halusinasi": {{
    "penilaian": "<Ya | Tidak>",
    "alasan": "<Jika 'Ya', sebutkan fakta yang diciptakan.>"
  }}
}}

"""

In [ ]:
# Inisialisasi Klien API (dilakukan sekali di luar loop/fungsi)
client = OpenAI(api_key=user_secrets.get_secret("API_DEEPSEEK"), base_url="https://api.deepseek.com")

# --- FUNGSI WORKER YANG DIMODIFIKASI ---
def evaluate_row(row_data):
    """
    Fungsi ini memproses SATU baris data, memanggil API, dan mengembalikan
    HASIL EVALUASI yang DIGABUNGKAN dengan DATA INPUT ASLI.
    """
    index, row = row_data  # Membongkar tuple dari iterrows()
    
    # --- PERUBAHAN 1: Siapkan dictionary data asli untuk disimpan ---
    original_data = {
        'Questions': row['Questions'],
        'Reference Answer': row['Reference Answer'],
        'Generate Answer': row['Generate Answer']
    }
    
    # Mengisi template prompt dengan data dari baris saat ini
    prompt = prompt_template.format(
        question=row['Questions'],
        reference_answer=row['Reference Answer'],
        generate_answer=row['Generate Answer']
    )
    
    try:
        # Panggil API
        response = client.chat.completions.create(
            model="deepseek-chat", 
            messages=[
                {"role": "system", "content": "Anda adalah evaluator AI ahli hukum yang hanya menghasilkan output JSON dalam Bahasa Indonesia."},
                {"role": "user", "content": prompt}
            ]
        )
        response_content = response.choices[0].message.content
        
        # Penanganan JSON yang kuat
        try:
            if "```json" in response_content:
                clean_json_str = response_content.split("```json\n")[1].split("\n```")[0]
            else:
                clean_json_str = response_content
            
            evaluation_result = json.loads(clean_json_str)
            # --- PERUBAHAN 2: Gabungkan data asli dengan hasil evaluasi ---
            return original_data | evaluation_result
            
        except (json.JSONDecodeError, IndexError) as parse_error:
            error_dict = {"error": f"JSON parsing failed on row {index}", "raw_response": response_content}
            # --- PERUBAHAN 3: Gabungkan data asli dengan dictionary error ---
            return original_data | error_dict
        
    except Exception as api_error:
        error_dict = {"error": f"API call failed on row {index}: {str(api_error)}"}
        # --- PERUBAHAN 3 (Lanjutan): Gabungkan data asli dengan dictionary error ---
        return original_data | error_dict


In [ ]:
# --- EKSEKUSI PARALEL ---
MAX_WORKERS = 100
evaluations = []

print(f"Memulai evaluasi paralel dengan {MAX_WORKERS} worker untuk {len(df)} baris data...")

with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
    results_iterator = executor.map(evaluate_row, df.iterrows())
    evaluations = list(tqdm(results_iterator, total=len(df), desc="Mengevaluasi (Paralel)"))


# --- ANALISIS HASIL ---
print("\nEvaluasi selesai. Memulai analisis...")

# DataFrame sekarang akan berisi kolom asli DAN kolom evaluasi
eval_df = pd.DataFrame(evaluations)
print("Contoh dari hasil evaluasi gabungan:")
print(eval_df.head())

# Simpan hasil lengkap untuk inspeksi manual yang lebih mudah
eval_df.to_csv('deepseek_evaluation_results_full.csv', index=False)
print("\nHasil evaluasi lengkap (termasuk data input) disimpan ke 'deepseek_evaluation_results_full.csv'")

# Summary

In [ ]:
if 'error' in eval_df.columns:
    # Filter baris di mana kolom 'error' adalah null (NaN)
    eval_df_successful = eval_df[eval_df['error'].isnull()].copy()
else:
    # Jika kolom 'error' tidak ada, berarti semua evaluasi berhasil.
    print("Tidak ditemukan kolom 'error', diasumsikan semua evaluasi berhasil.")
    eval_df_successful = eval_df.copy()

# Lanjutkan analisis hanya jika ada data yang berhasil
if not eval_df_successful.empty:
    print(f"\n--- Ringkasan Agregat dari {len(eval_df_successful)} Evaluasi yang Berhasil ---")
    try:
        # 'json_normalize' meratakan kolom yang berisi JSON. Ini adalah langkah kunci.
        # Ini akan membuat kolom baru seperti 'relevansi.skor', 'halusinasi.penilaian', dll.
        flat_eval_df = pd.json_normalize(eval_df_successful.to_dict('records'))

        # --- 1. Agregasi Metrik Skor (Rata-rata) ---
        print("\n[Metrik Skor Rata-rata (Skala 1-5)]")
        
        # Metrik Relevansi
        skor_relevansi_rata_rata = flat_eval_df['relevansi.skor'].mean()
        print(f"- Skor Relevansi Rata-rata:               {skor_relevansi_rata_rata:.2f}")

        # Metrik Akurasi
        skor_akurasi_rata_rata = flat_eval_df['akurasi_terhadap_sumber.skor'].mean()
        print(f"- Skor Akurasi vs. Sumber Rata-rata:      {skor_akurasi_rata_rata:.2f}")
        
        # Metrik Kelengkapan (BARU)
        skor_kelengkapan_rata_rata = flat_eval_df['kelengkapan_terhadap_sumber.skor'].mean()
        print(f"- Skor Kelengkapan vs. Sumber Rata-rata:  {skor_kelengkapan_rata_rata:.2f}")


        # --- 2. Agregasi Metrik Kategorikal (Distribusi Persentase) ---
        print("\n[Distribusi Penilaian Kategorikal]")

        # Metrik Akurasi Kutipan Pasal
        print("\n- Distribusi Akurasi Kutipan Pasal:")
        citation_counts = flat_eval_df['akurasi_kutipan_pasal.penilaian'].value_counts(normalize=True).mul(100)
        # Menampilkan dengan format yang rapi
        for category, percentage in citation_counts.items():
            print(f"  - {category:<20}: {percentage:.2f}%")

        # Metrik Halusinasi (BARU)
        print("\n- Distribusi Halusinasi:")
        hallucination_dist = flat_eval_df['halusinasi.penilaian'].value_counts(normalize=True).mul(100)
        for category, percentage in hallucination_dist.items():
            print(f"  - {category:<20}: {percentage:.2f}%")
            
        # Metrik khusus yang sangat penting: Tingkat Halusinasi
        # Dihitung sebagai persentase jawaban dengan penilaian 'Ya'
        hallucination_rate = hallucination_dist.get('Ya', 0.0) # .get() aman jika tidak ada 'Ya'
        print(f"\n  -> Tingkat Halusinasi (persentase 'Ya'): {hallucination_rate:.2f}%")


    except KeyError as e:
        print(f"\n[ERROR] Gagal saat menganalisis: Kolom yang diharapkan tidak ditemukan -> {e}")
        print("Ini kemungkinan besar karena output JSON dari model tidak konsisten atau tidak sesuai dengan template.")
        print("Silakan periksa file 'deepseek_evaluation_results.csv' untuk melihat output yang bermasalah.")
else:
    print("\nTidak ada evaluasi yang berhasil, analisis dilewati.")